# Demos de Concurrencia — 750001C Sistemas Operativos

**Autor:** Manuel Alejandro Pastrana Pardo, PhD. </br>
**Curso:** Sistemas Operativos — EISC, Universidad del Valle </br>
**Tema:** Race conditions, Lock, Productor-Consumidor, Filósofos comensales

Este notebook contiene el código ejecutable para las demostraciones en vivo de la sesión
*"De Hilos a Concurrencia"*. Continúa directamente el ejercicio de la sesión anterior
(vector de 10,000 elementos, 8 hilos, cálculo del cuadrado de cada elemento).

---


## 1. Setup — vector base y configuración

Mismo vector y configuración de la sesión de Hilos: 10,000 elementos, 8 hilos,
`chunk = 10000 // 8 = 1250`.

In [ ]:
import threading
import time
import random

NUM_HILOS = 8
TAM_VECTOR = 10_000
vector = list(range(TAM_VECTOR))

print(f"Vector de {TAM_VECTOR} elementos listo.")
print(f"NUM_HILOS = {NUM_HILOS}  ->  chunk = {TAM_VECTOR // NUM_HILOS}")


## 2. Demo (S5) — La race condition en vivo

Cada hilo calcula la suma de los cuadrados de su porción del vector y la **acumula
directamente** en la variable global `total` con `total += parcial`.

`total += parcial` NO es una operación atómica: internamente son tres pasos
(`load`, `add`, `store`). Si dos hilos hacen estos tres pasos entrelazados,
una actualización se puede "perder".

**Ejecuta esta celda varias veces.** Observa que `total` no siempre da el mismo valor.

In [ ]:
import sys

def sumar_sin_lock_sin_gil(hilo_id, vector):
    global total
    chunk = len(vector) // NUM_HILOS
    low, high = hilo_id * chunk, (hilo_id + 1) * chunk
    for i in range(low, high):
        temp = total                   # LOAD
        temp = temp + vector[i] ** 2   # ADD
        time.sleep(0)                 # <- cede el GIL aquí, garantizado
        total = temp                   # STORE  <-- race condition aquí

sys.setswitchinterval(1e-6)            # forzar cambios de contexto frecuentes

for intento in range(1, 6):
    total = 0
    hilos = [threading.Thread(target=sumar_sin_lock_sin_gil, args=(i, vector)) for i in range(NUM_HILOS)]
    for h in hilos: h.start()
    for h in hilos: h.join()
    print(f"Intento {intento}: total = {total}")

sys.setswitchinterval(0.005)  

In [ ]:
# algo importante de mencionar es que gracias al GIL de CPython solo cede el control entre hilos 
# cada sys.getswitchinterval() segundos (por defecto 5 ms). 
# Con NUM_HILOS=8 y chunk=1250, cada hilo termina su bucle completo en mucho menos de 5 ms.
# Por lo tanto, nunca le da tiempo al scheduler de interrumpir a un hilo justo entre el LOAD y el STORE de otro. 
# Si hacemos una implementación normal no se evidenciará la condición de carrera, porque cada hilo terminará su sección crítica sin interrupciones.
total = 0

def sumar_sin_lock(hilo_id, vector):
    """Cada hilo suma el cuadrado de su porción y acumula en 'total' SIN protección."""
    global total
    chunk = len(vector) // NUM_HILOS
    low, high = hilo_id * chunk, (hilo_id + 1) * chunk
    for i in range(low, high):
        total += vector[i] ** 2   # <-- race condition aquí: load -> add -> store


# Ejecutar varios intentos para evidenciar el no-determinismo
for intento in range(1, 6):
    total = 0
    hilos = [threading.Thread(target=sumar_sin_lock, args=(i, vector)) for i in range(NUM_HILOS)]

    for h in hilos:
        h.start()
    for h in hilos:
        h.join()

    print(f"Intento {intento}: total = {total}")


## 3. Valor de referencia (versión secuencial)

Calculamos el valor correcto de forma secuencial, sin hilos, para tener un punto de
comparación. Compáralo contra los resultados de la celda anterior.

In [ ]:
total_esperado = sum(v ** 2 for v in vector)
print(f"Valor correcto (secuencial): {total_esperado}")


## 4. Demo (S13 / S21) — La solución con `Lock`

Cada hilo calcula su `parcial` de forma independiente (esto SÍ se hace en paralelo,
sin ningún lock). Solo la actualización de `total` —la sección crítica— queda
protegida con `lock`.

**Ejecuta esta celda varias veces.** Ahora `total` siempre debe coincidir con el
valor esperado de la celda anterior.

In [ ]:
total = 0
lock = threading.Lock()

def sumar_con_lock(hilo_id, vector):
    """Cada hilo calcula su parcial en paralelo; solo el += está protegido."""
    global total
    chunk = len(vector) // NUM_HILOS
    low, high = hilo_id * chunk, (hilo_id + 1) * chunk

    parcial = sum(v ** 2 for v in vector[low:high])  # trabajo pesado, fuera del lock

    with lock:           # sección crítica: lo más pequeña posible
        total += parcial


for intento in range(1, 6):
    total = 0
    hilos = [threading.Thread(target=sumar_con_lock, args=(i, vector)) for i in range(NUM_HILOS)]

    for h in hilos:
        h.start()
    for h in hilos:
        h.join()

    correcto = "OK" if total == total_esperado else "ERROR"
    print(f"Intento {intento}: total = {total}   [{correcto}]")


## 5. Demo (S22) — Comparación de las 3 versiones

Comparamos correctitud y tiempo de ejecución de tres estrategias:

1. **Sin lock** — `total += parcial` directo (incorrecta, no determinística)
2. **Con lock** — `total += parcial` protegido (correcta, pero serializada)
3. **Vector de resultados independientes** — cada hilo escribe en su propia
   posición `resultados[hilo_id]`; se suma al final en el hilo principal
   (correcta, sin necesidad de lock)

**Ejecuta esta celda en clase** y completa la tabla con los valores reales obtenidos.

In [ ]:
import sys

def sumar_sin_lock_sin_gil(hilo_id, vector):
    global total
    chunk = len(vector) // NUM_HILOS
    low, high = hilo_id * chunk, (hilo_id + 1) * chunk
    for i in range(low, high):
        temp = total                   # LOAD
        temp = temp + vector[i] ** 2   # ADD
        time.sleep(0)                  # <- cede el GIL aquí, garantizado
        total = temp                   # STORE  <-- race condition aquí

sys.setswitchinterval(1e-6)            # forzar cambios de contexto frecuentes

for intento in range(1, 6):
    total = 0
    hilos = [threading.Thread(target=sumar_sin_lock_sin_gil, args=(i, vector)) for i in range(NUM_HILOS)]
    for h in hilos: h.start()
    for h in hilos: h.join()
    print(f"Intento {intento}: total = {total}")

sys.setswitchinterval(0.005)


def version_sin_lock():
    global total
    total = 0
    sys.setswitchinterval(1e-6)
    inicio = time.perf_counter()
    hilos = [threading.Thread(target=sumar_sin_lock_sin_gil, args=(i, vector)) for i in range(NUM_HILOS)]
    for h in hilos: h.start()
    for h in hilos: h.join()
    fin = time.perf_counter()
    sys.setswitchinterval(0.005)
    return total, fin - inicio


def version_con_lock():
    global total
    total = 0
    lock_local = threading.Lock()

    def trabajo(hilo_id):
        global total
        chunk = len(vector) // NUM_HILOS
        low, high = hilo_id * chunk, (hilo_id + 1) * chunk
        parcial = sum(v ** 2 for v in vector[low:high])
        with lock_local:
            total += parcial

    inicio = time.perf_counter()
    hilos = [threading.Thread(target=trabajo, args=(i,)) for i in range(NUM_HILOS)]
    for h in hilos: h.start()
    for h in hilos: h.join()
    fin = time.perf_counter()
    return total, fin - inicio


def version_vector_independiente():
    resultados = [0] * NUM_HILOS

    def trabajo(hilo_id):
        chunk = len(vector) // NUM_HILOS
        low, high = hilo_id * chunk, (hilo_id + 1) * chunk
        resultados[hilo_id] = sum(v ** 2 for v in vector[low:high])

    inicio = time.perf_counter()
    hilos = [threading.Thread(target=trabajo, args=(i,)) for i in range(NUM_HILOS)]
    for h in hilos: h.start()
    for h in hilos: h.join()
    fin = time.perf_counter()
    return sum(resultados), fin - inicio


# ── Ejecutar y mostrar tabla comparativa ───────────────────────────────────
print(f"Valor esperado: {total_esperado}\n")
print(f"{'Versión':<32} {'Resultado':>15} {'¿Correcto?':>12} {'Tiempo (s)':>12}")
print("-" * 75)

for nombre, funcion in [
    ("1. Sin lock",                version_sin_lock),
    ("2. Con lock",                version_con_lock),
    ("3. Vector independiente",    version_vector_independiente),
]:
    resultado, tiempo = funcion()
    correcto = "Sí" if resultado == total_esperado else "NO <-"
    print(f"{nombre:<32} {resultado:>15} {correcto:>12} {tiempo:>12.6f}")

## 6. Bonus — Productor-Consumidor con `Condition` (S16)

Un productor coloca elementos en un buffer de capacidad limitada. Un consumidor
los retira. `Condition` permite que cada hilo se duerma (`wait()`) hasta que el
otro le avise (`notify()`) que la situación cambió — sin *busy waiting*.

Nota el uso de `while` (no `if`) antes de `wait()`: al despertar, el hilo debe
re-verificar la condición.

In [ ]:
condicion = threading.Condition()
buffer = []
CAPACIDAD_MAX = 5
N_ITEMS = 10


def productor():
    for item in range(N_ITEMS):
        with condicion:
            while len(buffer) >= CAPACIDAD_MAX:
                print("  [Productor] buffer lleno, espera...")
                condicion.wait()
            buffer.append(item)
            print(f"[Productor] produjo {item}  (buffer={buffer})")
            condicion.notify()
        time.sleep(random.uniform(0.01, 0.05))


def consumidor():
    for _ in range(N_ITEMS):
        with condicion:
            while len(buffer) == 0:
                print("  [Consumidor] buffer vacío, espera...")
                condicion.wait()
            item = buffer.pop(0)
            print(f"[Consumidor] consumió {item}  (buffer={buffer})")
            condicion.notify()
        time.sleep(random.uniform(0.01, 0.05))


t_prod = threading.Thread(target=productor)
t_cons = threading.Thread(target=consumidor)

t_prod.start()
t_cons.start()
t_prod.join()
t_cons.join()

print("\nProductor-Consumidor terminado correctamente.")


## 7. Bonus — La Cena de los Filósofos (S10 / S11)

### 7.1 Versión con deadlock

Los 5 filósofos toman primero su tenedor **izquierdo** y luego el **derecho**.
Si los 5 toman el izquierdo al mismo tiempo, todos quedan esperando el derecho:
**deadlock**.

Para que el notebook no se quede colgado indefinidamente, usamos:

- `threading.Timer` — después de 3 segundos imprime un aviso de deadlock detectado
- `daemon=True` en los hilos — para que el notebook pueda continuar aunque los
  hilos sigan bloqueados en segundo plano

**Ejecuta esta celda y observa cómo, tras 3 segundos, ningún filósofo logra comer.**

In [ ]:
N_FILOSOFOS = 5
tenedores = [threading.Lock() for _ in range(N_FILOSOFOS)]
filosofos_que_comieron = set()


def filosofo_deadlock(i):
    izq = tenedores[i]
    der = tenedores[(i + 1) % N_FILOSOFOS]

    print(f"Filósofo {i} piensa...")
    time.sleep(0.05)

    print(f"Filósofo {i} toma tenedor izquierdo ({i})")
    izq.acquire()

    time.sleep(0.1)  # da tiempo a que todos tomen su tenedor izquierdo

    print(f"Filósofo {i} intenta tomar tenedor derecho ({(i + 1) % N_FILOSOFOS})...")
    der.acquire()    # <-- aquí se queda esperando para siempre (deadlock)

    print(f"Filósofo {i} ¡está comiendo!")
    filosofos_que_comieron.add(i)

    der.release()
    izq.release()


def aviso_deadlock():
    if len(filosofos_que_comieron) == 0:
        print("\n⚠ DEADLOCK detectado — ningún filósofo pudo comer.")
        print("  Los 5 hilos quedaron esperando un tenedor que nunca se libera.")
    else:
        print(f"\nFilósofos que lograron comer: {sorted(filosofos_que_comieron)}")


# Reiniciar los locks (por si esta celda se ejecuta más de una vez)
tenedores = [threading.Lock() for _ in range(N_FILOSOFOS)]
filosofos_que_comieron = set()

hilos = [threading.Thread(target=filosofo_deadlock, args=(i,), daemon=True)
         for i in range(N_FILOSOFOS)]

for h in hilos:
    h.start()

# Avisar tras 3 segundos sin necesidad de esperar indefinidamente
timer = threading.Timer(3.0, aviso_deadlock)
timer.start()
timer.join()


### 7.2 Versión corregida — romper la espera circular

Se rompe la simetría: el último filósofo (`i == N_FILOSOFOS - 1`) toma primero
el tenedor **derecho** y luego el izquierdo. Esto elimina el ciclo de espera
circular (condición #4 de Coffman) y el deadlock desaparece.

**Ejecuta esta celda.** Todos los filósofos deben lograr comer.

In [ ]:
def filosofo_correcto(i):
    izq = tenedores[i]
    der = tenedores[(i + 1) % N_FILOSOFOS]

    # El último filósofo invierte el orden de adquisición -> rompe la espera circular
    if i == N_FILOSOFOS - 1:
        primero, segundo = der, izq
    else:
        primero, segundo = izq, der

    print(f"Filósofo {i} piensa...")
    time.sleep(0.05)

    primero.acquire()
    segundo.acquire()

    print(f"Filósofo {i} ¡está comiendo!")
    filosofos_que_comieron.add(i)
    time.sleep(0.05)

    segundo.release()
    primero.release()


# Reiniciar locks y estado
tenedores = [threading.Lock() for _ in range(N_FILOSOFOS)]
filosofos_que_comieron = set()

hilos = [threading.Thread(target=filosofo_correcto, args=(i,)) for i in range(N_FILOSOFOS)]

for h in hilos:
    h.start()
for h in hilos:
    h.join()

print(f"\nFilósofos que lograron comer: {sorted(filosofos_que_comieron)}")
print("Sin deadlock — el orden de adquisición rompió la espera circular." )


---
## Resumen de lo demostrado

| # | Demo | Mecanismo | Conexión con la sesión |
|---|---|---|---|
| 2 | Race condition sin protección | — | Cierra el ejercicio de la sesión de Hilos |
| 4 | Solución con `Lock` | `threading.Lock` | Bloque 4.1 |
| 5 | Comparación 3 versiones | `Lock` vs. diseño sin estado compartido | Bloque 5.3 — la gran lección |
| 6 | Productor-Consumidor | `threading.Condition` | Bloque 4.4 / Bloque 3.1 |
| 7 | Filósofos — deadlock y solución | Orden de adquisición | Bloque 3.3 / 3.4 |
